# Module 1.2: Reasoning Patterns for Agentic AI

## Learning Objectives
By completing this notebook, you will:
1. Implement Chain of Thought (CoT) reasoning to improve agent decision quality
2. Apply self-consistency techniques to validate agent outputs across multiple samples
3. Build Tree of Thought systems for exploring multiple reasoning paths
4. Compare reasoning pattern effectiveness for different task types
5. Design hybrid reasoning strategies for complex multi-step problems

## Prerequisites
- Module 1.1: System Prompt Design
- Understanding of LLM sampling parameters (temperature, top_p)
- Basic probability and statistical concepts

## Estimated Time: 60 minutes

---

## Why Reasoning Patterns Matter

Standard LLM prompting generates answers in a single forward pass. This works for simple tasks but fails for:

- **Complex multi-step problems** requiring intermediate calculations
- **Tasks with multiple valid approaches** that need exploration
- **Ambiguous scenarios** requiring validation and verification

**Reasoning patterns** structure the agent's thinking process to:
1. **Break down complexity** into manageable steps
2. **Increase accuracy** through explicit reasoning chains
3. **Enable self-correction** by exposing intermediate logic
4. **Improve interpretability** for human oversight

**Real-world application:** Production agents use reasoning patterns for medical diagnosis, financial analysis, legal research, and technical troubleshooting - domains where wrong answers have consequences.

**Key insight:** Making the agent's reasoning visible and structured is more important than raw model capability.

## Setup & Imports

In [ ]:
# Purpose: Import required libraries for reasoning pattern implementations
# Key Concept: Separate imports for different reasoning strategies

import os
import json
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass
from collections import Counter
import re
from openai import OpenAI
from pydantic import BaseModel, Field

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Model configurations
REASONING_MODEL = "gpt-4o-mini"
DETERMINISTIC_TEMP = 0.0
CREATIVE_TEMP = 0.7
MAX_TOKENS = 2000

print("✓ Environment configured for reasoning pattern experiments")

## Part 1: Chain of Thought (CoT) Reasoning

Chain of Thought prompting instructs the agent to "think step by step" before answering. This exposes intermediate reasoning and significantly improves accuracy on complex tasks.

**Two CoT variants:**
1. **Zero-shot CoT:** Add "Let's think step by step" to the prompt
2. **Few-shot CoT:** Provide examples with reasoning chains

In [ ]:
# Purpose: Implement zero-shot Chain of Thought reasoning
# Key Concept: Simple prompt modification dramatically improves reasoning quality

def call_llm(prompt: str, temperature: float = DETERMINISTIC_TEMP) -> str:
    """
    Call OpenAI API with a user prompt.
    
    Parameters
    ----------
    prompt : str
        User message
    temperature : float
        Sampling temperature
    
    Returns
    -------
    str
        Model response
    """
    response = client.chat.completions.create(
        model=REASONING_MODEL,
        temperature=temperature,
        max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


def zero_shot_cot(question: str) -> Dict[str, str]:
    """
    Apply zero-shot Chain of Thought reasoning.
    
    Parameters
    ----------
    question : str
        Problem to solve
    
    Returns
    -------
    Dict[str, str]
        Contains 'reasoning' and 'answer' keys
    """
    # Step 1: Generate reasoning chain
    reasoning_prompt = f"{question}\n\nLet's think step by step:"
    reasoning = call_llm(reasoning_prompt)
    
    # Step 2: Extract final answer
    answer_prompt = f"{question}\n\nReasoning:\n{reasoning}\n\nBased on this reasoning, provide the final answer concisely:"
    answer = call_llm(answer_prompt)
    
    return {
        'reasoning': reasoning,
        'answer': answer
    }


# Test with a complex problem
problem = """
A company has 3 servers. Each server can handle 1000 requests per second.
Currently, requests arrive at 2500 per second during peak hours.
The company wants to maintain 20% capacity buffer for unexpected spikes.
How many additional servers are needed?
"""

print("Testing Zero-Shot Chain of Thought:")
print("=" * 60)
result = zero_shot_cot(problem)
print("\nReasoning Process:")
print(result['reasoning'])
print("\nFinal Answer:")
print(result['answer'])

In [ ]:
# Purpose: Implement few-shot Chain of Thought with examples
# Key Concept: Examples teach the agent the desired reasoning pattern

@dataclass
class CoTExample:
    """Example problem with reasoning chain and answer."""
    question: str
    reasoning: str
    answer: str


def few_shot_cot(question: str, examples: List[CoTExample]) -> Dict[str, str]:
    """
    Apply few-shot Chain of Thought with provided examples.
    
    Parameters
    ----------
    question : str
        Problem to solve
    examples : List[CoTExample]
        Example problems with reasoning chains
    
    Returns
    -------
    Dict[str, str]
        Contains 'reasoning' and 'answer' keys
    """
    # Build few-shot prompt
    prompt_parts = []
    
    for ex in examples:
        prompt_parts.append(f"Question: {ex.question}")
        prompt_parts.append(f"Reasoning: {ex.reasoning}")
        prompt_parts.append(f"Answer: {ex.answer}")
        prompt_parts.append("")
    
    prompt_parts.append(f"Question: {question}")
    prompt_parts.append("Reasoning:")
    
    full_prompt = "\n".join(prompt_parts)
    
    # Generate reasoning
    response = call_llm(full_prompt)
    
    # Extract reasoning and answer
    # The response should contain both reasoning continuation and answer
    parts = response.split("Answer:", 1)
    reasoning = parts[0].strip()
    answer = parts[1].strip() if len(parts) > 1 else "Answer not clearly separated"
    
    return {
        'reasoning': reasoning,
        'answer': answer
    }


# Create examples for capacity planning
examples = [
    CoTExample(
        question="A database can handle 500 queries/sec. Current load is 300 queries/sec. Need 30% buffer. Is current capacity sufficient?",
        reasoning="""1. Current capacity: 500 queries/sec
2. Current load: 300 queries/sec
3. Required buffer: 30% of capacity = 0.3 × 500 = 150 queries/sec
4. Effective capacity with buffer: 500 - 150 = 350 queries/sec
5. Compare: 300 queries/sec (load) < 350 queries/sec (effective capacity)""",
        answer="Yes, current capacity is sufficient. Load is 300 queries/sec while effective capacity (with buffer) is 350 queries/sec."
    )
]

print("\nTesting Few-Shot Chain of Thought:")
print("=" * 60)
result = few_shot_cot(problem, examples)
print("\nReasoning Process:")
print(result['reasoning'])
print("\nFinal Answer:")
print(result['answer'])

## Part 2: Self-Consistency - Validation Through Sampling

Self-consistency improves accuracy by:
1. Generating multiple reasoning paths (with temperature > 0)
2. Extracting final answers from each path
3. Taking the majority vote

**When to use:** Problems with clear correct answers where multiple reasoning approaches exist.

In [ ]:
# Purpose: Implement self-consistency through multiple sampling
# Key Concept: Majority voting across diverse reasoning paths increases robustness

def extract_final_answer(response: str) -> str:
    """
    Extract the final answer from a reasoning chain.
    
    Parameters
    ----------
    response : str
        Full reasoning response
    
    Returns
    -------
    str
        Extracted answer (normalized)
    """
    # Look for common answer patterns
    patterns = [
        r"(?:final answer|answer|therefore|thus)\s*:?\s*([^\n.]+)",
        r"(?:need|requires?)\s+(\d+)\s+(?:more|additional)?",
        r"(\d+)\s+(?:additional|more|extra)\s+servers?"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            return match.group(1).strip().lower()
    
    # Fallback: take last sentence
    sentences = response.strip().split('.')
    return sentences[-1].strip().lower() if sentences else response[:50].lower()


def self_consistency_cot(
    question: str,
    num_samples: int = 5,
    temperature: float = CREATIVE_TEMP
) -> Dict[str, Any]:
    """
    Apply self-consistency with multiple CoT samples.
    
    Parameters
    ----------
    question : str
        Problem to solve
    num_samples : int
        Number of reasoning paths to generate
    temperature : float
        Sampling temperature for diversity
    
    Returns
    -------
    Dict[str, Any]
        Contains 'samples', 'answers', 'majority_answer', 'confidence'
    """
    # Step 1: Generate multiple reasoning paths
    prompt = f"{question}\n\nLet's think step by step and provide a final answer:"
    samples = []
    answers = []
    
    for _ in range(num_samples):
        response = call_llm(prompt, temperature=temperature)
        answer = extract_final_answer(response)
        samples.append(response)
        answers.append(answer)
    
    # Step 2: Find majority answer
    answer_counts = Counter(answers)
    majority_answer, majority_count = answer_counts.most_common(1)[0]
    confidence = majority_count / num_samples
    
    return {
        'samples': samples,
        'answers': answers,
        'answer_counts': dict(answer_counts),
        'majority_answer': majority_answer,
        'confidence': confidence
    }


# Test self-consistency
print("\nTesting Self-Consistency:")
print("=" * 60)
result = self_consistency_cot(problem, num_samples=5)

print(f"\nGenerated {len(result['samples'])} reasoning paths")
print(f"\nExtracted answers: {result['answers']}")
print(f"\nAnswer distribution: {result['answer_counts']}")
print(f"\nMajority answer: {result['majority_answer']}")
print(f"Confidence: {result['confidence']:.1%}")

print("\nExample reasoning path:")
print(result['samples'][0][:300] + "...")

## Part 3: Tree of Thought - Exploring Multiple Paths

Tree of Thought (ToT) extends CoT by:
1. Generating multiple "thought steps" at each stage
2. Evaluating which thoughts are most promising
3. Exploring the best paths further (like beam search)

**When to use:** Problems requiring strategic exploration (planning, game playing, creative tasks).

In [ ]:
# Purpose: Implement Tree of Thought reasoning with path exploration
# Key Concept: Deliberate exploration of reasoning tree before committing to answer

@dataclass
class ThoughtNode:
    """A node in the tree of thoughts."""
    thought: str
    score: float
    depth: int
    parent: Optional['ThoughtNode'] = None
    children: List['ThoughtNode'] = None
    
    def __post_init__(self):
        if self.children is None:
            self.children = []
    
    def get_path(self) -> List[str]:
        """Get full reasoning path from root to this node."""
        path = []
        node = self
        while node:
            path.append(node.thought)
            node = node.parent
        return list(reversed(path))


def generate_thoughts(
    problem: str,
    current_path: List[str],
    num_thoughts: int = 3
) -> List[str]:
    """
    Generate candidate next thoughts given current reasoning path.
    
    Parameters
    ----------
    problem : str
        Original problem
    current_path : List[str]
        Reasoning steps taken so far
    num_thoughts : int
        Number of candidate thoughts to generate
    
    Returns
    -------
    List[str]
        Candidate next reasoning steps
    """
    path_text = "\n".join(f"{i+1}. {step}" for i, step in enumerate(current_path))
    
    prompt = f"""Problem: {problem}

Current reasoning steps:
{path_text if path_text else '(starting)'}

Generate {num_thoughts} different possible next reasoning steps. Each should explore a different approach.
Format: One step per line, numbered."""
    
    response = call_llm(prompt, temperature=CREATIVE_TEMP)
    
    # Parse numbered thoughts
    thoughts = []
    for line in response.split('\n'):
        # Match numbered lines
        match = re.match(r'^\d+\.\s*(.+)$', line.strip())
        if match:
            thoughts.append(match.group(1))
    
    return thoughts[:num_thoughts]


def evaluate_thought(
    problem: str,
    path: List[str]
) -> float:
    """
    Evaluate how promising a reasoning path is.
    
    Parameters
    ----------
    problem : str
        Original problem
    path : List[str]
        Current reasoning path
    
    Returns
    -------
    float
        Score from 0-10 indicating path quality
    """
    path_text = "\n".join(f"{i+1}. {step}" for i, step in enumerate(path))
    
    prompt = f"""Problem: {problem}

Reasoning path:
{path_text}

Evaluate this reasoning path on a scale of 0-10:
- Is it logically sound?
- Is it progressing toward a solution?
- Are the steps clear and justified?

Respond with just a number from 0-10."""
    
    response = call_llm(prompt, temperature=0.0)
    
    # Extract score
    match = re.search(r'(\d+(?:\.\d+)?)', response)
    return float(match.group(1)) if match else 5.0


def tree_of_thought(
    problem: str,
    max_depth: int = 3,
    num_thoughts: int = 3,
    beam_width: int = 2
) -> Dict[str, Any]:
    """
    Apply Tree of Thought reasoning with beam search.
    
    Parameters
    ----------
    problem : str
        Problem to solve
    max_depth : int
        Maximum reasoning depth
    num_thoughts : int
        Thoughts to generate at each node
    beam_width : int
        Number of best paths to keep exploring
    
    Returns
    -------
    Dict[str, Any]
        Contains 'best_path', 'best_score', 'all_paths'
    """
    # Initialize root
    root = ThoughtNode(thought="Start", score=10.0, depth=0)
    active_nodes = [root]
    all_paths = []
    
    # Explore tree level by level
    for depth in range(max_depth):
        next_nodes = []
        
        for node in active_nodes:
            # Generate candidate thoughts
            current_path = node.get_path()[1:]  # Exclude "Start"
            thoughts = generate_thoughts(problem, current_path, num_thoughts)
            
            # Create and evaluate child nodes
            for thought in thoughts:
                new_path = current_path + [thought]
                score = evaluate_thought(problem, new_path)
                
                child = ThoughtNode(
                    thought=thought,
                    score=score,
                    depth=depth + 1,
                    parent=node
                )
                node.children.append(child)
                next_nodes.append(child)
                all_paths.append((new_path, score))
        
        # Keep only top beam_width paths
        next_nodes.sort(key=lambda n: n.score, reverse=True)
        active_nodes = next_nodes[:beam_width]
        
        if not active_nodes:
            break
    
    # Find best path overall
    all_paths.sort(key=lambda x: x[1], reverse=True)
    best_path, best_score = all_paths[0] if all_paths else ([], 0.0)
    
    # Generate final answer from best path
    if best_path:
        path_text = "\n".join(f"{i+1}. {step}" for i, step in enumerate(best_path))
        answer_prompt = f"""{problem}

Based on this reasoning:
{path_text}

Provide the final answer:"""
        answer = call_llm(answer_prompt)
    else:
        answer = "No valid reasoning path found"
    
    return {
        'best_path': best_path,
        'best_score': best_score,
        'answer': answer,
        'num_paths_explored': len(all_paths)
    }


# Test Tree of Thought (simplified for demonstration)
print("\nTesting Tree of Thought:")
print("=" * 60)
print("(Running with reduced parameters for speed)\n")

tot_result = tree_of_thought(
    problem,
    max_depth=2,
    num_thoughts=2,
    beam_width=2
)

print(f"Explored {tot_result['num_paths_explored']} reasoning paths")
print(f"\nBest path (score: {tot_result['best_score']:.1f}):")
for i, step in enumerate(tot_result['best_path'], 1):
    print(f"{i}. {step}")
print(f"\nFinal Answer:\n{tot_result['answer']}")

## Part 4: Comparing Reasoning Patterns

Different patterns excel at different tasks. Let's compare them systematically.

In [ ]:
# Purpose: Benchmark different reasoning patterns on the same problem
# Key Concept: Pattern selection should match problem characteristics

def compare_reasoning_patterns(
    problem: str,
    ground_truth: Optional[str] = None
) -> Dict[str, Any]:
    """
    Compare all reasoning patterns on a single problem.
    
    Parameters
    ----------
    problem : str
        Problem to solve
    ground_truth : Optional[str]
        Known correct answer for comparison
    
    Returns
    -------
    Dict[str, Any]
        Results from each pattern
    """
    results = {}
    
    # Direct prompting (baseline)
    print("Running direct prompting...")
    direct = call_llm(problem)
    results['direct'] = {'answer': direct}
    
    # Zero-shot CoT
    print("Running zero-shot CoT...")
    zero_shot = zero_shot_cot(problem)
    results['zero_shot_cot'] = zero_shot
    
    # Self-consistency
    print("Running self-consistency...")
    self_consist = self_consistency_cot(problem, num_samples=3)
    results['self_consistency'] = {
        'answer': self_consist['majority_answer'],
        'confidence': self_consist['confidence']
    }
    
    return results


# Compare on our capacity planning problem
print("\nComparing Reasoning Patterns:")
print("=" * 60)
comparison = compare_reasoning_patterns(problem)

print("\nDirect Prompting:")
print(comparison['direct']['answer'][:200] + "...")

print("\n" + "-" * 60)
print("\nZero-Shot CoT:")
print(comparison['zero_shot_cot']['answer'][:200] + "...")

print("\n" + "-" * 60)
print("\nSelf-Consistency:")
print(f"Answer: {comparison['self_consistency']['answer']}")
print(f"Confidence: {comparison['self_consistency']['confidence']:.1%}")

## Exercises

### Exercise 2.1: Implement Custom CoT Extractor

**Task:** Build a function that extracts the reasoning chain and final answer separately from a CoT response.

**Requirements:**
- Parse responses with "Answer:" or "Therefore:" separators
- Extract numbered reasoning steps
- Handle cases where format is inconsistent

**Expected Output:** Dictionary with 'reasoning_steps' (list) and 'final_answer' (str)

In [ ]:
# YOUR CODE HERE
# ---------------

def parse_cot_response(response: str) -> Dict[str, Any]:
    """
    Extract structured reasoning and answer from CoT response.
    
    Parameters
    ----------
    response : str
        Raw CoT response
    
    Returns
    -------
    Dict[str, Any]
        Keys: 'reasoning_steps' (List[str]), 'final_answer' (str)
    """
    # Replace with your implementation
    return {
        'reasoning_steps': [],
        'final_answer': ""
    }


# Test data
test_response = """Let's solve this step by step:
1. Current capacity is 3 servers × 1000 = 3000 requests/sec
2. Current demand is 2500 requests/sec
3. With 20% buffer, need capacity for 2500 / 0.8 = 3125 requests/sec
4. Shortage is 3125 - 3000 = 125 requests/sec
5. Need 125 / 1000 = 0.125 servers, round up to 1

Answer: 1 additional server is needed."""

# Uncomment to test:
# parsed = parse_cot_response(test_response)
# print("Reasoning Steps:")
# for step in parsed['reasoning_steps']:
#     print(f"  - {step}")
# print(f"\nFinal Answer: {parsed['final_answer']}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_1():
    test_response = """Steps:
1. First calculation
2. Second step
Answer: Final result"""
    
    result = parse_cot_response(test_response)
    assert isinstance(result, dict), "Should return a dictionary"
    assert 'reasoning_steps' in result, "Should have 'reasoning_steps' key"
    assert 'final_answer' in result, "Should have 'final_answer' key"
    assert isinstance(result['reasoning_steps'], list), "Reasoning steps should be a list"
    assert len(result['reasoning_steps']) > 0, "Should extract reasoning steps"
    assert isinstance(result['final_answer'], str), "Final answer should be a string"
    assert len(result['final_answer']) > 0, "Should extract final answer"
    
    print("✅ Exercise 2.1 passed!")

test_exercise_2_1()

### Exercise 2.2: Build a Self-Consistency Scorer

**Task:** Create a function that measures agreement between multiple reasoning paths.

**Requirements:**
- Calculate answer diversity (how many unique answers)
- Compute agreement score (0-1 based on majority size)
- Identify outlier answers that disagree with majority

**Expected Output:** Dictionary with metrics about answer consistency

In [ ]:
# YOUR CODE HERE
# ---------------

def analyze_self_consistency(
    answers: List[str]
) -> Dict[str, Any]:
    """
    Analyze consistency metrics for multiple answers.
    
    Parameters
    ----------
    answers : List[str]
        List of extracted answers
    
    Returns
    -------
    Dict[str, Any]
        Keys: 'diversity', 'agreement_score', 'majority_answer', 'outliers'
    """
    # Replace with your implementation
    return {
        'diversity': 0,
        'agreement_score': 0.0,
        'majority_answer': "",
        'outliers': []
    }


# Uncomment to test:
# test_answers = ["2 servers", "2 servers", "2 servers", "1 server", "2 servers"]
# analysis = analyze_self_consistency(test_answers)
# print(analysis)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_2():
    test_answers = ["A", "A", "A", "B", "A"]
    result = analyze_self_consistency(test_answers)
    
    assert isinstance(result, dict), "Should return a dictionary"
    assert 'diversity' in result, "Should have 'diversity' key"
    assert 'agreement_score' in result, "Should have 'agreement_score' key"
    assert 'majority_answer' in result, "Should have 'majority_answer' key"
    assert result['diversity'] == 2, "Should count 2 unique answers"
    assert result['agreement_score'] >= 0.7, "Should show high agreement (4/5)"
    assert result['majority_answer'] == "A", "Should identify 'A' as majority"
    
    print("✅ Exercise 2.2 passed!")

test_exercise_2_2()

### Exercise 2.3: Implement Reasoning Pattern Selector

**Task:** Build a function that recommends the best reasoning pattern for a given problem.

**Requirements:**
- Analyze problem characteristics (length, complexity, domain)
- Return recommended pattern with justification
- Consider trade-offs (accuracy vs. speed vs. cost)

**Expected Output:** Pattern name and reasoning for recommendation

In [ ]:
# YOUR CODE HERE
# ---------------

def select_reasoning_pattern(
    problem: str,
    priority: str = "accuracy"  # "accuracy", "speed", or "cost"
) -> Dict[str, str]:
    """
    Recommend reasoning pattern for a problem.
    
    Parameters
    ----------
    problem : str
        Problem description
    priority : str
        Optimization target
    
    Returns
    -------
    Dict[str, str]
        Keys: 'pattern', 'reasoning'
    """
    # Replace with your implementation
    return {
        'pattern': 'zero_shot_cot',
        'reasoning': ''
    }


# Uncomment to test:
# problems = [
#     "What is 2+2?",
#     "Design a database schema for an e-commerce platform",
#     "If a train leaves Chicago at 2pm going 60mph..."
# ]
# for prob in problems:
#     rec = select_reasoning_pattern(prob, priority="accuracy")
#     print(f"Problem: {prob[:50]}...")
#     print(f"Recommended: {rec['pattern']}")
#     print(f"Reasoning: {rec['reasoning']}\n")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_3():
    simple_problem = "What is 2+2?"
    complex_problem = "Design a distributed system architecture for a social media platform with 1 billion users."
    
    result_simple = select_reasoning_pattern(simple_problem, priority="speed")
    result_complex = select_reasoning_pattern(complex_problem, priority="accuracy")
    
    assert isinstance(result_simple, dict), "Should return a dictionary"
    assert 'pattern' in result_simple, "Should have 'pattern' key"
    assert 'reasoning' in result_simple, "Should have 'reasoning' key"
    assert result_simple['pattern'] in ['direct', 'zero_shot_cot', 'few_shot_cot', 'self_consistency', 'tree_of_thought'], "Should return valid pattern"
    assert len(result_simple['reasoning']) > 20, "Should provide reasoning"
    
    # Complex problems should generally use more sophisticated patterns
    assert result_complex['pattern'] != 'direct', "Complex problem should not use direct prompting"
    
    print("✅ Exercise 2.3 passed!")

test_exercise_2_3()

### Exercise 2.4: Build a Hybrid Reasoning System

**Task:** Combine multiple reasoning patterns for improved robustness.

**Requirements:**
1. Use CoT to generate initial reasoning
2. Use self-consistency to validate (3 samples)
3. If confidence < 70%, escalate to tree of thought
4. Return final answer with confidence score and reasoning path

**Expected Output:** Comprehensive result with multi-stage reasoning

In [ ]:
# YOUR CODE HERE
# ---------------

def hybrid_reasoning(
    problem: str,
    confidence_threshold: float = 0.7
) -> Dict[str, Any]:
    """
    Apply hybrid reasoning with fallback strategies.
    
    Parameters
    ----------
    problem : str
        Problem to solve
    confidence_threshold : float
        Minimum confidence before escalation
    
    Returns
    -------
    Dict[str, Any]
        Keys: 'answer', 'confidence', 'method_used', 'reasoning_path'
    """
    # Replace with your implementation
    return {
        'answer': "",
        'confidence': 0.0,
        'method_used': "",
        'reasoning_path': []
    }


# Uncomment to test:
# test_problem = "A company has 100 employees. 40% work remotely. Of remote workers, 60% are in different time zones. How many employees are remote in different time zones?"
# result = hybrid_reasoning(test_problem)
# print(f"Answer: {result['answer']}")
# print(f"Confidence: {result['confidence']:.1%}")
# print(f"Method: {result['method_used']}")
# print(f"Reasoning path: {len(result['reasoning_path'])} steps")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_4():
    simple_problem = "If a store has 10 apples and sells 3, how many remain?"
    result = hybrid_reasoning(simple_problem)
    
    assert isinstance(result, dict), "Should return a dictionary"
    assert 'answer' in result, "Should have 'answer' key"
    assert 'confidence' in result, "Should have 'confidence' key"
    assert 'method_used' in result, "Should have 'method_used' key"
    assert 'reasoning_path' in result, "Should have 'reasoning_path' key"
    assert 0 <= result['confidence'] <= 1, "Confidence should be between 0 and 1"
    assert isinstance(result['reasoning_path'], list), "Reasoning path should be a list"
    
    print("✅ Exercise 2.4 passed!")

test_exercise_2_4()

## Summary

**Key Takeaways:**

1. **Chain of Thought** exposes intermediate reasoning, dramatically improving complex problem-solving
2. **Self-consistency** validates answers through multiple reasoning paths and majority voting
3. **Tree of Thought** explores multiple strategies before committing to an answer
4. **Pattern selection** depends on problem type, accuracy requirements, and resource constraints
5. **Hybrid approaches** combine strengths of multiple patterns for production robustness

**Performance characteristics:**
- **Direct prompting:** Fast, low cost, but unreliable for complex tasks
- **Zero-shot CoT:** Minimal overhead, significant accuracy improvement
- **Self-consistency:** High accuracy, 3-5x API cost
- **Tree of Thought:** Best for exploration, 10-20x API cost

**What's Next:**
- Module 1.3: Prompt Optimization - Few-shot learning, templates, dynamic prompting
- Module 2: Tool Use & Function Calling - Extending agents with external capabilities

**Additional Resources:**
- [Chain of Thought Prompting Elicits Reasoning in LLMs (Wei et al., 2022)](https://arxiv.org/abs/2201.11903)
- [Self-Consistency Improves Chain of Thought (Wang et al., 2022)](https://arxiv.org/abs/2203.11171)
- [Tree of Thoughts: Deliberate Problem Solving (Yao et al., 2023)](https://arxiv.org/abs/2305.10601)